# 📧 Spam Message Detection System
**End-to-end ML pipeline** — data loading → preprocessing → feature extraction → model training → evaluation → prediction

---
**Dataset:** SMS Spam Collection (UCI / Kaggle) — 5,574 messages labeled `spam` or `ham`  
**Models:** Naive Bayes · Logistic Regression · Random Forest · Linear SVM

## 1️⃣  Install & Import Dependencies

In [ ]:
# Run once
!pip install -q scikit-learn nltk pandas numpy matplotlib seaborn wordcloud imbalanced-learn

In [ ]:
import re, string, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
import joblib

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
print('✅ All libraries imported successfully')

## 2️⃣  Load Dataset

In [ ]:
# Option A: Load from local file
# df = pd.read_csv('spam.csv', encoding='latin-1')[['v1','v2']]
# df.columns = ['label','message']

# Option B: Auto-download from GitHub mirror
import urllib.request, io
url = 'https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv'
try:
    with urllib.request.urlopen(url) as r:
        df = pd.read_csv(io.StringIO(r.read().decode()), sep='\t',
                         header=None, names=['label','message'])
    print('✅ Dataset downloaded successfully')
except Exception:
    print('⚠️  Network unavailable — using built-in demo data')
    df = pd.DataFrame({
        'label': ['ham','ham','spam','spam','ham','spam','ham','spam','ham','spam',
                  'ham','spam','ham','ham','spam','spam','ham','ham','spam','ham'],
        'message': [
            'Hey, are you free tomorrow for lunch?',
            'I will be late for the meeting, sorry.',
            'WINNER!! You have been selected for a $1000 Walmart gift card. Call NOW!',
            'Congratulations! You won a FREE iPhone. Claim at http://scam.net',
            'Can you pick up some groceries on your way home?',
            'URGENT: Your account has been compromised. Verify at http://phish.com',
            'The project deadline is next Friday.',
            'FREE entry to win $5000 cash. Text WIN to 88888 now!',
            'Mom wants to know if you are coming for dinner.',
            'You have been pre-approved for a $10,000 loan! No credit check!',
            'Just checking in — how are you doing?',
            'Click here to claim your FREE prize worth $500!',
            'The meeting is moved to 3 PM.',
            'Happy birthday! Hope you have a wonderful day!',
            'FINAL NOTICE: Collect your reward before it expires at http://win.com',
            'You are a lucky winner! Call 0800-FREE to claim your prize.',
            'I will call you back in 10 minutes.',
            'See you at the gym this evening.',
            'Limited time offer! Buy 1 get 2 FREE. Visit us NOW!',
            'Let me know when you are available for a quick call.'
        ]
    })

df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})
print(f'Dataset shape: {df.shape}')
print(df['label'].value_counts())
df.head()

## 3️⃣  Exploratory Data Analysis (EDA)

In [ ]:
df['msg_len'] = df['message'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class balance
df['label'].value_counts().plot(kind='bar', ax=axes[0],
                                color=['steelblue','tomato'], edgecolor='white')
axes[0].set_title('Class Distribution', fontsize=13)
axes[0].tick_params(rotation=0)
for p in axes[0].patches:
    axes[0].annotate(str(int(p.get_height())),
                     (p.get_x()+p.get_width()/2, p.get_height()+10), ha='center')

# Message length
df.groupby('label')['msg_len'].plot(kind='hist', ax=axes[1], bins=30, alpha=0.6,
                                    color=['steelblue','tomato'])
axes[1].set_title('Message Length Distribution', fontsize=13)
axes[1].set_xlabel('Characters')
axes[1].legend(['ham','spam'])

plt.tight_layout()
plt.show()

In [ ]:
df['num_words']  = df['message'].apply(lambda x: len(x.split()))
df['num_urls']   = df['message'].apply(lambda x: len(re.findall(r'http\S+|www\.\S+', x)))
df['num_digits'] = df['message'].apply(lambda x: sum(c.isdigit() for c in x))
df['num_caps']   = df['message'].apply(lambda x: sum(c.isupper() for c in x))

feat_cols = ['msg_len','num_words','num_urls','num_digits','num_caps']
print(df.groupby('label')[feat_cols].mean().round(2))

plt.figure(figsize=(7, 4))
sns.heatmap(df[feat_cols + ['label_num']].corr(), annot=True,
            fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, lbl, title, cmap in zip(axes,
                                 ['ham','spam'],
                                 ['Ham Messages','Spam Messages'],
                                 ['Blues','Reds']):
    text = ' '.join(df[df['label']==lbl]['message'])
    wc   = WordCloud(width=600, height=300, colormap=cmap,
                     background_color='white').generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')
plt.suptitle('Word Clouds', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣  Text Preprocessing

In [ ]:
stemmer    = PorterStemmer()
STOP_WORDS = set(stopwords.words('english'))

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', 'url', text)       # URLs → 'url'
    text = re.sub(r'\S+@\S+', 'email', text)               # emails → 'email'
    text = re.sub(r'\b\d[\d\s\-\.]{7,}\d\b', 'phone', text) # phones → 'phone'
    text = re.sub(r'[^a-z\s]', ' ', text)                  # remove non-alpha
    tokens = text.split()
    tokens = [stemmer.stem(t) for t in tokens
               if t not in STOP_WORDS and len(t) > 1]
    return ' '.join(tokens)

df['clean_message'] = df['message'].apply(preprocess_text)

print('=== Before & After ===')
for _, row in df.sample(4, random_state=1).iterrows():
    print(f'[{row["label"].upper()}]')
    print(f'  Original : {row["message"]}')
    print(f'  Cleaned  : {row["clean_message"]}')
    print()

## 5️⃣  Train / Test Split + TF-IDF Vectorization

In [ ]:
X = df['clean_message']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2),
                        sublinear_tf=True, min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Vocabulary size   : {len(tfidf.vocabulary_)}')
print(f'Train matrix shape: {X_train_tfidf.shape}')

## 6️⃣  Train Multiple Models

In [ ]:
MODELS = {
    'Naive Bayes'         : MultinomialNB(alpha=0.1),
    'Logistic Regression' : LogisticRegression(C=5, max_iter=1000, class_weight='balanced'),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42,
                                                    class_weight='balanced', n_jobs=-1),
    'Linear SVM'          : LinearSVC(C=1.0, class_weight='balanced', max_iter=2000),
}

results = {}
for name, clf in MODELS.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    acc   = accuracy_score(y_test, preds)
    try:
        scores = clf.predict_proba(X_test_tfidf)[:, 1]
    except AttributeError:
        score_raw = clf.decision_function(X_test_tfidf)
        scores = 1 / (1 + np.exp(-score_raw))
    auc = roc_auc_score(y_test, scores)
    results[name] = {'model': clf, 'preds': preds, 'scores': scores,
                     'acc': acc, 'auc': auc}
    print(f'{name:<25}  Acc={acc:.4f}  AUC={auc:.4f}')

## 7️⃣  Evaluation — Classification Reports

In [ ]:
for name, res in results.items():
    print(f'\n{"="*55}')
    print(f'  {name}')
    print('='*55)
    print(classification_report(y_test, res['preds'], target_names=['ham','spam']))

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5*len(MODELS), 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['ham','spam'], yticklabels=['ham','spam'])
    ax.set_title(f'{name}\nAcc={res["acc"]:.3f}', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for (name, res), color in zip(results.items(),
                               ['steelblue','tomato','seagreen','mediumpurple']):
    fpr, tpr, _ = roc_curve(y_test, res['scores'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", lw=2, color=color)
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
summary_df = pd.DataFrame(
    [{'Model': k, 'Accuracy': v['acc'], 'AUC-ROC': v['auc']}
     for k, v in results.items()]
).sort_values('AUC-ROC', ascending=False)

x = np.arange(len(summary_df)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, summary_df['Accuracy'], w, label='Accuracy', color='steelblue', alpha=0.85)
b2 = ax.bar(x + w/2, summary_df['AUC-ROC'],  w, label='AUC-ROC',  color='tomato',   alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(summary_df['Model'], rotation=15, ha='right')
ax.set_ylim(0.9, 1.01); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold'); ax.legend()
for bar in list(b1)+list(b2):
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x()+bar.get_width()/2, bar.get_height()),
                xytext=(0,3), textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout(); plt.show()
print(summary_df.to_string(index=False))

## 8️⃣  Feature Importance (Logistic Regression)

In [ ]:
lr_model   = results['Logistic Regression']['model']
coef       = lr_model.coef_[0]
feat_names = tfidf.get_feature_names_out()

top_spam = np.argsort(coef)[-20:][::-1]
top_ham  = np.argsort(coef)[:20]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, idx, title, color in [
    (axes[0], top_spam, 'Top 20 SPAM indicators', 'tomato'),
    (axes[1], top_ham,  'Top 20 HAM  indicators', 'steelblue')
]:
    ax.barh(feat_names[idx][::-1], coef[idx][::-1], color=color, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Coefficient Weight')
    ax.axvline(0, color='black', lw=0.8)
plt.tight_layout(); plt.show()

## 9️⃣  Cross-Validation

In [ ]:
print('5-Fold Cross-Validation (F1 Score)\n' + '='*45)
for name, clf in MODELS.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                                   sublinear_tf=True, min_df=2)),
        ('clf', clf)
    ])
    scores = cross_val_score(pipe, X, y, cv=5, scoring='f1', n_jobs=-1)
    print(f'{name:<25}  F1 = {scores.mean():.4f} ± {scores.std():.4f}')

## 🔟  Save Best Model

In [ ]:
best_name = max(results, key=lambda k: results[k]['auc'])
best_clf  = results[best_name]['model']
print(f'🏆 Best model: {best_name}  (AUC={results[best_name]["auc"]:.4f})')

joblib.dump(best_clf, 'spam_model.pkl')
joblib.dump(tfidf,    'tfidf_vectorizer.pkl')
print('✅ Saved: spam_model.pkl')
print('✅ Saved: tfidf_vectorizer.pkl')

## 1️⃣1️⃣  Prediction Function & Live Demo

In [ ]:
def predict_spam(message: str, clf=best_clf, vectorizer=tfidf) -> dict:
    clean = preprocess_text(message)
    vec   = vectorizer.transform([clean])
    pred  = clf.predict(vec)[0]
    try:
        prob = clf.predict_proba(vec)[0][1]
    except AttributeError:
        s    = clf.decision_function(vec)[0]
        prob = 1 / (1 + np.exp(-s))
    conf = prob if pred == 1 else 1 - prob
    return {'label': 'SPAM 🚫' if pred else 'HAM  ✅',
            'confidence': f'{conf*100:.1f}%'}


test_msgs = [
    'WINNER!! You have been selected for a $1000 gift card. Click here to claim!',
    'Hey, are we still meeting for lunch tomorrow?',
    'Congratulations! You have won a FREE ticket to Bahamas. Reply YES to claim.',
    'Can you send me the report by end of day?',
    'URGENT: Your bank account requires verification. Call 0800-FREE now!',
    'Looking forward to seeing you at the conference next week.',
]

print(f'{"Message":<55} {"Label":<12} Confidence')
print('-'*80)
for msg in test_msgs:
    r = predict_spam(msg)
    short = (msg[:52]+'...') if len(msg) > 52 else msg
    print(f'{short:<55} {r["label"]:<12} {r["confidence"]}')

## 1️⃣2️⃣  FastAPI Deployment Snippet

Save as `api.py` and run with `uvicorn api:app --reload`:

In [ ]:
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np

app  = FastAPI(title="Spam Detector API")
clf  = joblib.load("spam_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

class Message(BaseModel):
    text: str

@app.post("/predict")
def predict(msg: Message):
    clean = preprocess_text(msg.text)     # import preprocess_text from preprocessing module
    vec   = tfidf.transform([clean])
    pred  = clf.predict(vec)[0]
    try:
        prob = clf.predict_proba(vec)[0][1]
    except AttributeError:
        s    = clf.decision_function(vec)[0]
        prob = 1 / (1 + np.exp(-s))
    return {"label": "spam" if pred else "ham",
            "confidence": round(float(prob), 4)}
'''
print(api_code)

## 1️⃣3️⃣  Next Steps & Improvements

| Improvement | How |
|---|---|
| **BERT / Transformers** | Fine-tune `distilbert-base-uncased` with HuggingFace Trainer |
| **Handle imbalance** | SMOTE from `imbalanced-learn` or `class_weight='balanced'` |
| **Stacking ensemble** | Combine NB + LR + SVM with a meta-classifier |
| **REST API** | Wrap `predict_spam()` in FastAPI (see cell above) |
| **Richer features** | URL ratio, caps ratio, emoji count as numeric features |
| **Multilingual** | `langdetect` + language-specific stopword lists |
| **Active learning** | Retrain on flagged real-world messages |
